In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from ipywidgets import FloatSlider, RadioButtons, VBox, HBox, HTML, interactive_output, Layout
from IPython.display import display

# ============================================================
# SURFACE SAMPLING IN A UNIFORM QUANTIZER
# ============================================================

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="font-family:Arial, sans-serif; font-size:16px; line-height:1.42; width:1080px;">

<div style="font-size:22px; font-weight:bold; color:#76448a; margin-bottom:8px;">
Surface Sampling of a Probability Density by a Quantizer
</div>

<div style="margin-bottom:4px;">
A uniform quantizer maps every input interval of width Δ to a single output level y = mΔ.
</div>

<div style="margin-bottom:4px;">
The probability assigned to each output level is not a sample of fₓ(x), but the <b>area</b> of fₓ(x) over the corresponding quantization interval.
</div>

<div style="margin-bottom:4px;">
Thus P{Y = mΔ} = ∫ fₓ(x) dx over the interval [mΔ−Δ/2, mΔ+Δ/2].
</div>

<div>
<b>This notebook:</b> visualizes how a continuous input PDF is transformed into a discrete output probability distribution by surface sampling.
</div>

</div>
""")

# ============================================================
# INPUT DISTRIBUTION SELECTOR
# ============================================================

distribution_selector = RadioButtons(options=['Gaussian', 'Uniform'], value='Gaussian', description='', layout=Layout(width='180px'))

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '105px'}

delta_slider = FloatSlider(min=0.25, max=2.00, step=0.25, value=1.00, description='Step Δ:', continuous_update=True, readout=True, readout_format='.2f', style=slider_style, layout=Layout(width='270px'))

mu_slider = FloatSlider(min=-1.00, max=1.00, step=0.10, value=0.00, description='Mean μ:', continuous_update=True, readout=True, readout_format='.1f', style=slider_style, layout=Layout(width='270px'))

sigma_slider = FloatSlider(min=0.50, max=1.50, step=0.10, value=1.00, description='Std. dev. σ:', continuous_update=True, readout=True, readout_format='.1f', style=slider_style, layout=Layout(width='270px'))

# ============================================================
# CONTROL PANEL
# ============================================================

controls_card = VBox(
    [
        HTML('<div style="font-family:Arial; font-size:18px; font-weight:bold; color:#76448a; margin-bottom:8px;">Quantizer Parameters</div>'),
        HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; margin-bottom:3px;">Input distribution</div>'),
        distribution_selector,
        HTML('<div style="height:5px;"></div>'),
        delta_slider,
        mu_slider,
        sigma_slider
    ],
    layout=Layout(width='300px', min_width='300px', padding='12px 12px', border='1px solid #d8c4df', margin='12px 0px 0px 14px', overflow='hidden')
)

# ============================================================
# NUMERICAL RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# INPUT PDF
# ============================================================

def input_pdf(x, distribution, mu, sigma):

    if distribution == 'Gaussian':
        return norm.pdf(x, loc=mu, scale=sigma)

    half_width = np.sqrt(3.0) * sigma
    pdf = np.zeros_like(x)
    mask = (x >= mu - half_width) & (x <= mu + half_width)
    pdf[mask] = 1.0 / (2.0 * half_width)

    return pdf

# ============================================================
# BIN PROBABILITY
# ============================================================

def bin_probability(left, right, distribution, mu, sigma):

    if distribution == 'Gaussian':
        return norm.cdf(right, loc=mu, scale=sigma) - norm.cdf(left, loc=mu, scale=sigma)

    half_width = np.sqrt(3.0) * sigma
    support_left = mu - half_width
    support_right = mu + half_width

    overlap_left = max(left, support_left)
    overlap_right = min(right, support_right)

    if overlap_right <= overlap_left:
        return 0.0

    return (overlap_right - overlap_left) / (2.0 * half_width)

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_surface_sampling(distribution='Gaussian', delta=1.0, mu=0.0, sigma=1.0):

    # --------------------------------------------------------
    # FIXED DISPLAY RANGE
    # --------------------------------------------------------

    x_min = -6.0
    x_max = 6.0

    x = np.linspace(x_min, x_max, 2400)

    fx = input_pdf(x, distribution, mu, sigma)

    # --------------------------------------------------------
    # QUANTIZATION LEVELS
    # --------------------------------------------------------

    m_min = int(np.floor(x_min / delta)) - 1
    m_max = int(np.ceil(x_max / delta)) + 1

    m = np.arange(m_min, m_max + 1)

    levels = m * delta

    probabilities = np.zeros(len(levels))

    for i, level in enumerate(levels):
        left = level - delta / 2.0
        right = level + delta / 2.0
        probabilities[i] = bin_probability(left, right, distribution, mu, sigma)

    # --------------------------------------------------------
    # KEEP ONLY LEVELS INSIDE DISPLAY RANGE
    # --------------------------------------------------------

    visible = (levels >= x_min) & (levels <= x_max)

    visible_levels = levels[visible]
    visible_probabilities = probabilities[visible]

    # --------------------------------------------------------
    # QUANTIZER CHARACTERISTIC
    # --------------------------------------------------------

    xq = np.linspace(x_min, x_max, 2400)

    yq = delta * np.floor(xq / delta + 0.5)

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(8.2, 7.0))

    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.05], hspace=0.43, wspace=0.30)

    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    # ========================================================
    # GRAPH 1:
    # INPUT PDF AND QUANTIZATION INTERVALS
    # ========================================================

    ax1.plot(x, fx, linewidth=2.0, label='Input PDF fₓ(x)')

    for level, probability in zip(visible_levels, visible_probabilities):

        left = level - delta / 2.0
        right = level + delta / 2.0

        mask = (x >= left) & (x <= right)

        if np.any(mask):
            ax1.fill_between(x[mask], 0, fx[mask], alpha=0.18)

        ax1.axvline(left, linewidth=0.7, linestyle=':', alpha=0.6)

    ax1.axvline(visible_levels[-1] + delta / 2.0, linewidth=0.7, linestyle=':', alpha=0.6)

    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(0, 0.90)
    ax1.set_xlabel('Input amplitude x', fontsize=10)
    ax1.set_ylabel('Probability density fₓ(x)', fontsize=10)
    ax1.set_title('Input PDF Divided into Quantization Cells', fontsize=12, pad=8)
    ax1.tick_params(axis='both', labelsize=9)
    ax1.grid(True, linestyle=':', alpha=0.35)
    ax1.legend(loc='upper right', fontsize=8)

    # ========================================================
    # GRAPH 2:
    # QUANTIZER INPUT / OUTPUT CHARACTERISTIC
    # ========================================================

    ax2.step(xq, yq, where='mid', linewidth=1.7, label='y = Q(x)')
    ax2.plot(xq, xq, linestyle='--', linewidth=1.2, label='y = x')

    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(x_min, x_max)
    ax2.set_xlabel('Input x', fontsize=10)
    ax2.set_ylabel('Quantized output y', fontsize=10)
    ax2.set_title('Uniform Mid-Tread Quantizer', fontsize=12, pad=8)
    ax2.tick_params(axis='both', labelsize=9)
    ax2.grid(True, linestyle=':', alpha=0.35)
    ax2.legend(loc='upper left', fontsize=8)

    # ========================================================
    # GRAPH 3:
    # OUTPUT DISCRETE PROBABILITY DISTRIBUTION
    # ========================================================

    markerline, stemlines, baseline = ax3.stem(visible_levels, visible_probabilities, basefmt=' ')

    ax3.set_xlim(x_min, x_max)
    ax3.set_ylim(0, 1.05)
    ax3.set_xlabel('Quantization level y = mΔ', fontsize=10)
    ax3.set_ylabel('P{Y = mΔ}', fontsize=10)
    ax3.set_title('Quantizer Output Probabilities', fontsize=12, pad=8)
    ax3.tick_params(axis='both', labelsize=9)
    ax3.grid(True, linestyle=':', alpha=0.35)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(left=0.09, right=0.97, top=0.93, bottom=0.09)

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL STATISTICS
    # ========================================================

    probability_sum = np.sum(probabilities)

    mean_y = np.sum(levels * probabilities)

    variance_y = np.sum((levels - mean_y)**2 * probabilities)

    # Quantization MSE evaluated numerically from the input PDF
    qx = delta * np.floor(x / delta + 0.5)

    error = qx - x

    mse = np.trapezoid(error**2 * fx, x)

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.45;
        width:760px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Total output probability:</b>
    Σ P{{Y = mΔ}} = {probability_sum:.6f}

    <br>

    <b>Output mean:</b>
    E{{Y}} = {mean_y:.5f}

    &nbsp;&nbsp;&nbsp;

    <b>Output variance:</b>
    Var{{Y}} = {variance_y:.5f}

    <br>

    <b>Quantization mean-square error:</b>
    E{{(Y−X)²}} ≈ {mse:.6f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_surface_sampling,
    {
        'distribution': distribution_selector,
        'delta': delta_slider,
        'mu': mu_slider,
        'sigma': sigma_slider
    }
)

# ============================================================
# GRAPH AREA
# ============================================================

graph_area = VBox(
    [
        output,
        result_html
    ],
    layout=Layout(width='800px', min_width='800px', overflow='hidden')
)

# ============================================================
# MAIN BODY
#
# GRAPHS LEFT - CONTROLS RIGHT
# ============================================================

body_layout = HBox(
    [
        graph_area,
        controls_card
    ],
    layout=Layout(width='1120px', align_items='flex-start', justify_content='flex-start', overflow='hidden')
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1100px;
    padding:11px 15px;
    border:1px solid #d8cbe3;
    background:#fcf9ff;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#76448a;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
Each quantizer output level represents one interval of input amplitudes rather than one particular value of the input PDF.
</div>

<div style="margin-bottom:4px;">
The probability associated with level mΔ is the complete area of fₓ(x) inside the corresponding interval of width Δ.
</div>

<div>
Therefore quantization transforms a continuous probability density into a discrete probability distribution through <b>surface sampling</b>, not ordinary point sampling.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox([documentation, body_layout, interpretation], layout=Layout(width='1120px', overflow='hidden'))

display(main_layout)